In [11]:
"""
FCC Fe (Austenite) EBSD — ML Training Dataset Generator
========================================================
Generates a labelled EBSD pattern dataset for strain classification.

Pipeline:
    1. Build kinematical master patterns for each strain level
    2. Sample orientations uniformly from the FCC fundamental zone
    3. Project patterns onto a virtual detector per orientation × strain × noise
    4. Apply dynamic background removal
    5. Save to HDF5 with embedded train/val/test split indices

HDF5 structure:
    /patterns           float32  (N, H, W)
    /strain_pct         float32  (N,)
    /euler_angles_deg   float32  (N, 3)    Bunge convention
    /lattice_params     float32  (N, 3)    a, b, c in Å
    /noise_level        float32  (N,)
    /split              str      (N,)      "train" | "val" | "test"
    /idx_train          int64    (n_train,)
    /idx_val            int64    (n_val,)
    /idx_test           int64    (n_test,)

Usage:
    python fcc_fe_ebsd_ml_dataset.py

Requirements:
    pip install kikuchipy diffsims orix diffpy.structure h5py numpy matplotlib

References:
    Wilkinson et al., Ultramicroscopy 106:307 (2006)
    Marquardt et al., Ultramicroscopy 184:167 (2018)
    Rowenhorst et al., Modelling Simul. Mater. Sci. Eng. 23 (2015)
"""

import os
import json
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import h5py
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import kikuchipy as kp
from diffsims.crystallography import ReciprocalLatticeVector
from orix.crystal_map import Phase
from orix.quaternion import Rotation
from orix.sampling import get_sample_fundamental
from diffpy.structure import Atom, Lattice, Structure


# =============================================================================
# Configuration
# =============================================================================

CFG = {
    # Paths
    "project_dir":           "/Users/jfj3094/Documents/FP-465/MATSCI_465_local",
    "output_subdir":         "ml_dataset",

    # Crystal — FCC Fe (austenite), space group Fm-3m
    "a0":                    3.59,       # Å, reference lattice parameter
    "poisson":               0.29,       # Poisson ratio
    "space_group":           225,

    # Simulation
    "voltage_kev":           20,
    "min_dspacing":          0.7,        # Å
    "master_size":           256,        # master pattern half-size (px)
    "detector_shape":        (80, 80),   # px
    "pc":                    (0.42, 0.22, 0.50),  # projection centre (Bruker)
    "sample_tilt":           70.0,       # degrees

    # Orientations
    "n_orientations":        500,
    "resolution":            1.5,        # degrees — cubochoric sampling

    # Training strain levels (discrete, evenly spaced, cached for speed)
    "strain_max_pct":        50.0,
    "n_train_strain_levels": 20,

    # Test strain levels — 2 per class, drawn from bin centres
    #   Class 0 (0–5%):    2.0, 4.0
    #   Class 1 (5–15%):   7.5, 12.5
    #   Class 2 (15–30%):  20.0, 27.0
    #   Class 3 (30–50%):  37.5, 47.5
    "test_strains_pct":      [2.0, 4.0, 7.5, 12.5, 20.0, 27.0, 37.5, 47.5],

    # Background removal
    "dynamic_bg_std":        8,          # Gaussian sigma (px)

    # Noise — Poisson shot noise + Gaussian read noise
    "noise_levels":          [0.0, 0.01, 0.02, 0.05],
    "poisson_scale":         1e4,

    # Dataset split (by orientation, not pattern — prevents data leakage)
    "train_frac":            0.70,
    "val_frac":              0.15,

    # Reproducibility
    "seed":                  42,
}

OUTDIR  = os.path.join(CFG["project_dir"], CFG["output_subdir"])
H5_PATH = os.path.join(OUTDIR, "ebsd_fcc_fe.h5")
FIG_DIR = os.path.join(CFG["project_dir"], "figures")

rng = np.random.default_rng(CFG["seed"])
A0  = CFG["a0"]
NU  = CFG["poisson"]


# =============================================================================
# Crystal helpers
# =============================================================================

def make_phase(a, b=None, c=None, name="austenite"):
    b = a if b is None else b
    c = a if c is None else c
    return Phase(
        name=name,
        space_group=CFG["space_group"],
        structure=Structure(
            atoms=[Atom("Fe", [0, 0, 0])],
            lattice=Lattice(a, b, c, 90, 90, 90),
        ),
    )


def strained_abc(eps_pct):
    """Lattice parameters (a, b, c) in Å for uniaxial strain along [100]."""
    eps = eps_pct / 100.0
    return A0 * (1.0 + eps), A0 * (1.0 - NU * eps), A0 * (1.0 - NU * eps)


def deformation_gradient(eps_pct):
    """3×3 deformation gradient tensor F = diag(1+ε, 1−νε, 1−νε)."""
    eps = eps_pct / 100.0
    return np.diag([1 + eps, 1 - NU * eps, 1 - NU * eps])


# =============================================================================
# Master pattern (cached per strain level)
# =============================================================================

def build_master_pattern(phase):
    """Kinematical master pattern in Lambert projection for the given phase."""
    rlv = ReciprocalLatticeVector.from_min_dspacing(
        phase, min_dspacing=CFG["min_dspacing"]
    )
    rlv.sanitise_phase()
    rlv.calculate_structure_factor()
    rlv.calculate_theta(voltage=CFG["voltage_kev"] * 1e3)
    rlv = rlv[rlv.allowed]
    sim = kp.simulations.KikuchiPatternSimulator(rlv)
    return sim.calculate_master_pattern(half_size=CFG["master_size"]).as_lambert()


detector = kp.detectors.EBSDDetector(
    shape=CFG["detector_shape"],
    pc=CFG["pc"],
    sample_tilt=CFG["sample_tilt"],
)

_mp_cache: dict = {}


# =============================================================================
# Pattern simulation
# =============================================================================

def remove_background(sig):
    """Dynamic background removal via Gaussian subtraction."""
    sig.remove_dynamic_background(
        operation="subtract",
        std=CFG["dynamic_bg_std"],
        show_progressbar=False,
    )
    return sig


def get_clean_pattern(eps_pct, rotation):
    """
    Simulate a background-corrected EBSD pattern.

    Master patterns are built from the strained phase and cached by strain
    level so each unique eps_pct is only computed once.

    Returns
    -------
    pat : float32 ndarray, shape (H, W), normalised to [0, 1]
    """
    if eps_pct not in _mp_cache:
        a, b, c = strained_abc(eps_pct)
        _mp_cache[eps_pct] = build_master_pattern(
            make_phase(a, b, c, name=f"austenite_{eps_pct:.4f}pct")
        )

    sig = _mp_cache[eps_pct].get_patterns(
        rotations=rotation,
        detector=detector,
        energy=CFG["voltage_kev"],
        compute=True,
        show_progressbar=False,
    )

    try:
        sig = remove_background(sig)
    except Exception as e:
        print(f"  [WARN] Background removal skipped at ε={eps_pct:.2f}%: {e}")

    pat = sig.data[0].astype(np.float32)
    return (pat - pat.min()) / (pat.max() - pat.min() + 1e-9)


def add_noise(pat, sigma, rng):
    """
    Two-component noise model: Poisson shot noise + Gaussian read noise.

    Returns
    -------
    float32 ndarray clipped to [0, 1]
    """
    if sigma == 0.0:
        return pat.copy()
    counts  = pat * CFG["poisson_scale"]
    poisson = rng.normal(0, np.sqrt(np.maximum(counts, 1e-6))) / CFG["poisson_scale"]
    gauss   = rng.normal(0, sigma, pat.shape).astype(np.float32)
    return np.clip(pat + poisson + gauss, 0, 1).astype(np.float32)


# =============================================================================
# Orientation sampling
# =============================================================================

def sample_orientations(n, resolution_deg, seed):
    """
    Sample n rotations uniformly from the FCC (m-3m) fundamental zone
    using cubochoric sampling via orix.
    """
    from orix.quaternion.symmetry import get_point_group
    pg   = get_point_group(225, proper=False)
    full = get_sample_fundamental(
        resolution=resolution_deg, point_group=pg, method="cubochoric"
    )
    n_full    = full.size
    rng_local = np.random.default_rng(seed)
    if n_full >= n:
        idx = rng_local.choice(n_full, size=n, replace=False)
    else:
        print(f"  [WARN] Only {n_full} orientations at {resolution_deg}°; using all.")
        idx = np.arange(n_full)
    sampled = full[idx]
    print(f"  Orientations: {sampled.size} sampled from {n_full} in fundamental zone")
    return sampled


# =============================================================================
# Train / val / test split
# =============================================================================

def assign_splits(n_ori, train_f, val_f, seed):
    """
    Assign orientations to train/val/test by index shuffle.
    Test count is the exact remainder so that splits always sum to n_ori.
    """
    rng_local = np.random.default_rng(seed)
    idx     = rng_local.permutation(n_ori)
    n_train = int(n_ori * train_f)
    n_val   = int(n_ori * val_f)
    n_test  = n_ori - n_train - n_val
    splits  = {}
    for i, ori_i in enumerate(idx):
        if i < n_train:
            splits[ori_i] = "train"
        elif i < n_train + n_val:
            splits[ori_i] = "val"
        else:
            splits[ori_i] = "test"
    print(f"  Split: {n_train} train / {n_val} val / {n_test} test orientations")
    return splits


# =============================================================================
# Strain levels
# =============================================================================

def get_train_strain_levels():
    """
    Evenly-spaced training strain levels as Python floats.
    Float precision ensures exact cache key matches.
    """
    n  = CFG["n_train_strain_levels"]
    mx = CFG["strain_max_pct"]
    return [round(float(v), 6) for v in np.linspace(0.0, mx, n)]


# =============================================================================
# Dataset generation
# =============================================================================

def generate_dataset():
    t0 = time.time()

    train_levels = get_train_strain_levels()
    test_levels  = [float(s) for s in CFG["test_strains_pct"]]
    all_levels   = sorted(set(train_levels + test_levels))

    print(f"\n[1/5] Building master patterns for {len(all_levels)} strain levels...")
    for eps in all_levels:
        a, b, c = strained_abc(eps)
        _mp_cache[eps] = build_master_pattern(
            make_phase(a, b, c, name=f"austenite_{eps:.4f}pct")
        )
        print(f"  ε={eps:6.2f}%  a={a:.4f}  b={b:.4f} Å")
    print(f"  Done — {len(_mp_cache)} patterns cached ({time.time()-t0:.0f}s)")

    print("\n[2/5] Sampling orientations...")
    rotations = sample_orientations(CFG["n_orientations"], CFG["resolution"], CFG["seed"])
    n_ori     = rotations.size
    splits    = assign_splits(n_ori, CFG["train_frac"], CFG["val_frac"], CFG["seed"])

    n_strain      = len(train_levels)
    n_noise       = len(CFG["noise_levels"])
    n_noise_test  = 2
    n_test_strains = len(CFG["test_strains_pct"])
    n_train_ori   = sum(1 for v in splits.values() if v == "train")
    n_val_ori     = sum(1 for v in splits.values() if v == "val")
    n_test_ori    = sum(1 for v in splits.values() if v == "test")

    n_train = n_train_ori * n_strain * n_noise
    n_val   = n_val_ori   * n_strain * n_noise
    n_test  = n_test_ori  * n_test_strains * n_noise_test
    N_total = n_train + n_val + n_test
    H, W    = CFG["detector_shape"]

    print(f"\n[3/5] Dataset preview:")
    print(f"  Train : {n_train:>6}  ({n_train_ori} ori × {n_strain} strains × {n_noise} noise)")
    print(f"  Val   : {n_val:>6}  ({n_val_ori} ori × {n_strain} strains × {n_noise} noise)")
    print(f"  Test  : {n_test:>6}  ({n_test_ori} ori × {n_test_strains} strains × {n_noise_test} noise)")
    print(f"  Total : {N_total:>6}")

    print(f"\n[4/5] Simulating {N_total} patterns...")
    all_patterns = np.zeros((N_total, H, W), dtype=np.float32)
    all_strains  = np.zeros(N_total,         dtype=np.float32)
    all_eulers   = np.zeros((N_total, 3),    dtype=np.float32)
    all_lattice  = np.zeros((N_total, 3),    dtype=np.float32)
    all_noise    = np.zeros(N_total,         dtype=np.float32)
    all_splits   = np.empty(N_total,         dtype=object)
    idx_write    = 0

    for ori_i in range(n_ori):
        rot         = rotations[ori_i]
        split_label = splits[ori_i]
        euler       = rot.to_euler(degrees=True).flatten()[:3]

        if split_label in ("train", "val"):
            strain_vals = train_levels
            noise_vals  = CFG["noise_levels"]
        else:
            strain_vals = test_levels
            noise_vals  = [0.0, 0.01]

        for eps_pct in strain_vals:
            a, b, c   = strained_abc(float(eps_pct))
            clean_pat = get_clean_pattern(float(eps_pct), rot)

            for sigma in noise_vals:
                all_patterns[idx_write] = add_noise(clean_pat, sigma, rng)
                all_strains [idx_write] = eps_pct
                all_eulers  [idx_write] = euler
                all_lattice [idx_write] = [a, b, c]
                all_noise   [idx_write] = sigma
                all_splits  [idx_write] = split_label
                idx_write += 1

        if (ori_i + 1) % 50 == 0 or ori_i == n_ori - 1:
            pct = 100 * idx_write / N_total
            print(f"  {ori_i+1:>4}/{n_ori} orientations  |  {pct:5.1f}%  |  "
                  f"{time.time()-t0:.0f}s elapsed")

    all_patterns = all_patterns[:idx_write]
    all_strains  = all_strains [:idx_write]
    all_eulers   = all_eulers  [:idx_write]
    all_lattice  = all_lattice [:idx_write]
    all_noise    = all_noise   [:idx_write]
    all_splits   = all_splits  [:idx_write]

    print(f"\n[5/5] Writing HDF5 → {H5_PATH}")
    with h5py.File(H5_PATH, "w") as f:
        f.attrs["config"]    = json.dumps(CFG)
        f.attrs["generated"] = time.strftime("%Y-%m-%dT%H:%M:%S")
        f.attrs["n_total"]   = idx_write
        f.attrs["n_train"]   = int(np.sum(all_splits == "train"))
        f.attrs["n_val"]     = int(np.sum(all_splits == "val"))
        f.attrs["n_test"]    = int(np.sum(all_splits == "test"))

        f.create_dataset("patterns",         data=all_patterns,
                         compression="gzip", compression_opts=4,
                         chunks=(64, H, W))
        f.create_dataset("strain_pct",       data=all_strains)
        f.create_dataset("euler_angles_deg", data=all_eulers)
        f.create_dataset("lattice_params",   data=all_lattice)
        f.create_dataset("noise_level",      data=all_noise)
        f.create_dataset("split",            data=all_splits.astype("S8"))

        for sp in ("train", "val", "test"):
            f.create_dataset(
                f"idx_{sp}",
                data=np.where(all_splits == sp)[0].astype(np.int64)
            )

    size_mb = os.path.getsize(H5_PATH) / 1e6
    print(f"  Saved {idx_write} patterns  ({size_mb:.1f} MB)")


# =============================================================================
# QC figures
# =============================================================================

def make_qc_figures():
    print("\nGenerating QC figures...")

    with h5py.File(H5_PATH, "r") as f:
        strains      = f["strain_pct"][:]
        splits       = f["split"][:].astype(str)
        eulers       = f["euler_angles_deg"][:]
        noise        = f["noise_level"][:]
        idx_tr       = f["idx_train"][:]
        idx_va       = f["idx_val"][:]
        idx_te       = f["idx_test"][:]
        n_ex         = min(3, len(idx_tr))
        ex_pats      = f["patterns"][idx_tr[:n_ex]]
        ex_str       = strains[idx_tr[:n_ex]]
        ex_ns        = noise[idx_tr[:n_ex]]
        test_pats    = f["patterns"][idx_te]
        test_strains = strains[idx_te]

    plt.rcParams.update({
        "font.family":       "Arial",
        "axes.spines.top":   False,
        "axes.spines.right": False,
        "axes.labelsize":    10,
        "axes.titlesize":    10,
        "axes.titleweight":  "bold",
        "xtick.labelsize":   9,
        "ytick.labelsize":   9,
        "axes.grid":         True,
        "grid.alpha":        0.25,
        "grid.linestyle":    "--",
    })

    BLUE   = "#2E74B5"
    ORANGE = "#C55A11"
    GREEN  = "#4EA72A"
    NAVY   = "#1F497D"

    # ── Figure 1: QC summary ─────────────────────────────────────────────────
    fig = plt.figure(figsize=(16, 13))
    fig.suptitle("FCC Fe Austenite — EBSD ML Dataset Quality Control",
                 fontsize=13, fontweight="bold", y=0.98)
    gs = fig.add_gridspec(3, 3, hspace=0.55, wspace=0.38)

    ax_a = fig.add_subplot(gs[0, 0])
    ax_a.hist(strains[splits != "test"], bins=50, color=BLUE,
              edgecolor="white", linewidth=0.3)
    ax_a.set_xlabel("Strain (%)")
    ax_a.set_ylabel("Count")
    ax_a.set_title("A  Training Strain Distribution")
    ax_a.text(0.5, -0.28,
              "Uniform sampling across 0–50%.\n"
              "Each of the 20 discrete levels has equal representation.",
              transform=ax_a.transAxes, ha="center", va="top",
              fontsize=8, color="#444", style="italic")

    ax_b = fig.add_subplot(gs[0, 1])
    test_strain_vals = sorted(CFG["test_strains_pct"])
    test_counts = [
        int(np.sum(np.abs(strains[splits == "test"] - lvl) < 0.01))
        for lvl in test_strain_vals
    ]
    bars_b = ax_b.bar([f"{v:.1f}" for v in test_strain_vals], test_counts,
                      color=ORANGE, edgecolor="white")
    ax_b.set_xlabel("Strain (%)")
    ax_b.set_ylabel("Count")
    ax_b.tick_params(axis="x", labelsize=7.5, rotation=40)
    for bar, cnt in zip(bars_b, test_counts):
        ax_b.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                  str(cnt), ha="center", fontsize=7, fontweight="bold")
    ax_b.set_title("B  Test Set Strain Levels")
    ax_b.text(0.5, -0.28,
              "2 strain levels per class (0–5 / 5–15 / 15–30 / 30–50%).\n"
              "Balanced for fair per-class F1 evaluation.",
              transform=ax_b.transAxes, ha="center", va="top",
              fontsize=8, color="#444", style="italic")

    ax_c = fig.add_subplot(gs[0, 2])
    counts      = [len(idx_tr), len(idx_va), len(idx_te)]
    split_cols  = [BLUE, GREEN, ORANGE]
    bars_c = ax_c.bar(["Train", "Val", "Test"], counts,
                      color=split_cols, edgecolor="white", width=0.5)
    for bar, cnt in zip(bars_c, counts):
        ax_c.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
                  f"{cnt:,}", ha="center", fontsize=9, fontweight="bold")
    ax_c.set_ylabel("Patterns")
    ax_c.set_title("C  Dataset Split Sizes")
    ax_c.text(0.5, -0.28,
              "70 / 15 / 15% split by orientation cluster.\n"
              "Prevents data leakage across splits.",
              transform=ax_c.transAxes, ha="center", va="top",
              fontsize=8, color="#444", style="italic")

    ax_d = fig.add_subplot(gs[1, 0])
    stride            = max(1, CFG["n_train_strain_levels"] * len(CFG["noise_levels"]))
    tr_idx_ori_unique = idx_tr[::stride]
    ax_d.scatter(eulers[tr_idx_ori_unique, 0], eulers[tr_idx_ori_unique, 1],
                 s=6, alpha=0.65, color=NAVY, linewidths=0)
    ax_d.set_xlabel("φ₁ (°)")
    ax_d.set_ylabel("Φ (°)")
    ax_d.set_xlim(0, 360)
    ax_d.set_ylim(0, 65)
    ax_d.set_title("D  Orientation Coverage (train)")
    ax_d.text(0.5, -0.28,
              "Each point = one unique grain orientation (Bunge Euler angles).\n"
              "Even coverage across the FCC fundamental zone (m-3m).",
              transform=ax_d.transAxes, ha="center", va="top",
              fontsize=8, color="#444", style="italic")

    ax_e = fig.add_subplot(gs[1, 1])
    unique_noise, noise_counts = np.unique(noise[splits == "train"], return_counts=True)
    ax_e.bar([f"{v:.2f}" for v in unique_noise], noise_counts,
             color="#4472C4", edgecolor="white", width=0.5)
    ax_e.set_xlabel("Gaussian σ")
    ax_e.set_ylabel("Count")
    ax_e.set_title("E  Noise Level Distribution (train)")
    ax_e.text(0.5, -0.28,
              "Poisson shot noise + Gaussian read noise at 4 levels.\n"
              "Forces the model to learn strain signal, not noise artefacts.",
              transform=ax_e.transAxes, ha="center", va="top",
              fontsize=8, color="#444", style="italic")

    captions = ["Clean pattern (σ=0.00)", "Light noise (σ=0.01)", "Moderate noise (σ=0.02)"]
    for i in range(n_ex):
        ax = fig.add_subplot(gs[2, i])
        ax.imshow(ex_pats[i], cmap="gray", vmin=0, vmax=1)
        ax.set_title(f"ε = {ex_str[i]:.1f}%  |  σ = {ex_ns[i]:.2f}", fontsize=10, pad=4)
        ax.axis("off")
        ax.text(0.5, -0.10, captions[i], transform=ax.transAxes,
                ha="center", va="top", fontsize=8, color="#444", style="italic")

    fig.text(0.5, 0.005,
             "Figure F — Example EBSD Kikuchi patterns after dynamic background removal. "
             "Bands remain visible across all noise levels. "
             "The CNN classifies strain from these 80×80 px images.",
             ha="center", fontsize=8.5, color="#333", style="italic")

    fig.savefig(os.path.join(FIG_DIR, "dataset_qc.png"), dpi=150, bbox_inches="tight")
    print("  Saved → dataset_qc.png")
    plt.close(fig)

    # ── Figure 2: Patterns at each test strain level ──────────────────────────
    test_levels = sorted(CFG["test_strains_pct"])
    level_pats  = {}
    for lvl in test_levels:
        mask = (np.abs(test_strains - lvl) < 0.01) & (noise[idx_te] == 0.0)
        if not np.any(mask):
            mask = np.abs(test_strains - lvl) == np.abs(test_strains - lvl).min()
        level_pats[lvl] = test_pats[np.where(mask)[0][0]]

    ref_pat     = level_pats[test_levels[0]]
    bin_edges   = [0, 5, 15, 30, 50]
    class_names = ["Very Low", "Low", "Medium", "High"]
    class_cols  = {"Very Low": BLUE, "Low": GREEN, "Medium": ORANGE, "High": "#7030A0"}

    def get_class(eps):
        for ci, (lo, hi) in enumerate(zip(bin_edges, bin_edges[1:])):
            if lo <= eps <= hi:
                return class_names[ci]
        return ""

    n_levels = len(test_levels)
    fig2, axes2 = plt.subplots(2, n_levels, figsize=(2.8 * n_levels, 6),
                                gridspec_kw={"height_ratios": [1, 0.08]})
    fig2.suptitle("EBSD Kikuchi Patterns — Test Set (Clean)",
                  fontsize=11, fontweight="bold", y=1.01)

    for j, lvl in enumerate(test_levels):
        ax_img = axes2[0][j]
        ax_lbl = axes2[1][j]
        ax_img.imshow(level_pats[lvl], cmap="gray", vmin=0, vmax=1)
        ax_img.set_title(f"ε = {lvl:.1f}%", fontsize=9, pad=3)
        ax_img.axis("off")
        cls = get_class(lvl)
        ax_lbl.text(0.5, 0.5, cls, transform=ax_lbl.transAxes,
                    ha="center", va="center", fontsize=8,
                    color="white", fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.3",
                              facecolor=class_cols.get(cls, "gray"),
                              edgecolor="none"))
        ax_lbl.axis("off")

    fig2.text(0.5, -0.04,
              "Kikuchi bands shift as lattice d-spacings change with strain. "
              "Colour labels show the CNN strain class for each pattern.",
              ha="center", fontsize=8.5, color="#333", style="italic")
    fig2.tight_layout()
    fig2.savefig(os.path.join(FIG_DIR, "patterns_all_strains.png"),
                 dpi=150, bbox_inches="tight")
    print("  Saved → patterns_all_strains.png")
    plt.close(fig2)

    # ── Figure 3: Difference maps ─────────────────────────────────────────────
    non_zero = [lvl for lvl in test_levels if lvl > 0]
    n_diff   = len(non_zero)
    vmax_g   = max(np.abs(level_pats[lvl] - ref_pat).max() for lvl in non_zero)

    fig3, axes3 = plt.subplots(1, n_diff, figsize=(2.8 * n_diff, 4.5))
    fig3.suptitle("Kikuchi Band Shift Maps — Strained vs. Reference (ε = 0%)",
                  fontsize=11, fontweight="bold")

    for ax, lvl in zip(axes3, non_zero):
        dm  = level_pats[lvl] - ref_pat
        im  = ax.imshow(dm, cmap="RdBu_r", vmin=-vmax_g, vmax=vmax_g)
        mad = np.mean(np.abs(dm))
        ax.set_title(f"ε = {lvl:.1f}%", fontsize=9, pad=3)
        ax.text(0.5, -0.06, f"MAD = {mad:.4f}",
                transform=ax.transAxes, ha="center", fontsize=8, color="#444")
        ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    fig3.text(0.5, -0.04,
              "Red = intensity gain (band moved in); Blue = intensity loss (band moved out). "
              "MAD quantifies detectability — larger MAD = easier for the CNN to classify.",
              ha="center", fontsize=8.5, color="#333", style="italic")
    fig3.tight_layout()
    fig3.savefig(os.path.join(FIG_DIR, "diff_maps.png"), dpi=150, bbox_inches="tight")
    print("  Saved → diff_maps.png")
    plt.close(fig3)

    # ── Figure 4: Strain sensitivity ──────────────────────────────────────────
    mad_vals     = [np.mean(np.abs(level_pats[lvl] - ref_pat)) for lvl in non_zero]
    class_colors = ["#D6E8F7", "#D5ECD5", "#FDE8D8", "#EAD6F5"]
    class_labels = ["Very Low\n(0–5%)", "Low\n(5–15%)", "Medium\n(15–30%)", "High\n(30–50%)"]

    fig4, ax4 = plt.subplots(figsize=(8, 4.5))
    ax4.plot(non_zero, mad_vals, "o-", color=NAVY,
             linewidth=2, markersize=7, markerfacecolor=BLUE, markeredgecolor="white")
    for eps, mad in zip(non_zero, mad_vals):
        ax4.annotate(f"{mad:.4f}", (eps, mad),
                     textcoords="offset points", xytext=(6, 5), fontsize=8)
    for (lo, hi), col, lbl in zip(zip(bin_edges, bin_edges[1:]), class_colors, class_labels):
        ax4.axvspan(lo, hi, alpha=0.35, color=col, zorder=0)
        ax4.text((lo + hi) / 2, 0, lbl, ha="center", va="bottom",
                 fontsize=7.5, color="#555")
    ax4.set_xlabel("Applied Strain ε (%)", fontsize=11)
    ax4.set_ylabel("Mean Absolute Difference (MAD)", fontsize=11)
    ax4.set_title("EBSD Pattern Sensitivity to Strain", fontsize=11, fontweight="bold")
    ax4.set_xlim(0, 52)
    fig4.text(0.5, -0.06,
              "A monotonically increasing MAD confirms strain is physically encoded in the patterns. "
              "Higher strain produces more distinguishable Kikuchi band shifts.",
              ha="center", fontsize=8.5, color="#333", style="italic")
    fig4.tight_layout()
    fig4.savefig(os.path.join(FIG_DIR, "strain_sensitivity.png"),
                 dpi=150, bbox_inches="tight")
    print("  Saved → strain_sensitivity.png")
    plt.close(fig4)

    plt.rcParams.update(plt.rcParamsDefault)


# =============================================================================
# PyTorch dataset loader (written to output dir, no kikuchipy dependency)
# =============================================================================

PYTORCH_LOADER = '''\
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset


class EBSDStrainDataset(Dataset):
    """
    PyTorch Dataset for the FCC Fe EBSD strain classification dataset.

    Parameters
    ----------
    h5_path   : str   — path to ebsd_fcc_fe.h5
    split     : str   — "train" | "val" | "test"
    transform : optional torchvision transform

    Returns (per item)
    ------------------
    pat_t     : float32 tensor  (1, H, W)
    strain_t  : float32 scalar  strain in %
    euler_t   : float32 tensor  (3,) Bunge Euler angles in degrees
    """

    def __init__(self, h5_path, split="train", transform=None):
        self.h5_path   = h5_path
        self.split     = split
        self.transform = transform
        self._h5       = None

        with h5py.File(h5_path, "r") as f:
            self.indices = f[f"idx_{split}"][:].tolist()

    def _open(self):
        if self._h5 is None:
            self._h5 = h5py.File(self.h5_path, "r")

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        self._open()
        idx          = self.indices[item]
        pat          = self._h5["patterns"][idx]
        strain       = self._h5["strain_pct"][idx]
        euler        = self._h5["euler_angles_deg"][idx]
        pat_t        = torch.tensor(pat[None], dtype=torch.float32)
        if self.transform:
            pat_t = self.transform(pat_t)
        return pat_t, torch.tensor(strain), torch.tensor(euler)

    def __del__(self):
        if self._h5 is not None:
            self._h5.close()
'''


def write_pytorch_loader():
    path = os.path.join(OUTDIR, "pytorch_dataset.py")
    with open(path, "w") as f:
        f.write(PYTORCH_LOADER)
    print(f"  Saved → {path}")


# =============================================================================
# Entry point
# =============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("  FCC Fe EBSD — ML Dataset Generator")
    print(f"  Output   : {OUTDIR}")
    print(f"  Ori.     : {CFG['n_orientations']}")
    print(f"  Strains  : {CFG['n_train_strain_levels']} levels (0–{CFG['strain_max_pct']}%)")
    print(f"  Noise    : {CFG['noise_levels']}")
    print(f"  Seed     : {CFG['seed']}")
    print("=" * 60)

    generate_dataset()
    make_qc_figures()
    write_pytorch_loader()

    print("\n" + "=" * 60)
    print("  Done.")
    print(f"  Dataset : {H5_PATH}")
    print(f"  Figures : {FIG_DIR}/")
    print(f"  Loader  : {OUTDIR}/pytorch_dataset.py")
    print("=" * 60)


  FCC Fe EBSD — ML Dataset Generator
  Output   : /Users/jfj3094/Documents/FP-465/MATSCI_465_local/ml_dataset
  Ori.     : 500
  Strains  : 20 levels (0–50.0%)
  Noise    : [0.0, 0.01, 0.02, 0.05]
  Seed     : 42

[1/5] Building master patterns for 28 strain levels...


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00, 15.15it/s]

  ε=  0.00%  a=3.5900  b=3.5900 Å



100%|█████████████████████████████████████████████| 1/1 [00:00<00:00, 15.42it/s]


  ε=  2.00%  a=3.6618  b=3.5692 Å


...

  ε= 47.37%  a=5.2905  b=3.0968 Å


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00, 14.73it/s]


  ε= 47.50%  a=5.2953  b=3.0955 Å


100%|█████████████████████████████████████████████| 1/1 [00:00<00:00, 14.53it/s]


  ε= 50.00%  a=5.3850  b=3.0694 Å
  Done — 28 patterns cached (5s)

[2/5] Sampling orientations...
  Orientations: 500 sampled from 243129 in fundamental zone
  Split: 350 train / 75 val / 75 test orientations

[3/5] Dataset preview:
  Train :  28000  (350 ori × 20 strains × 4 noise)
  Val   :   6000  (75 ori × 20 strains × 4 noise)
  Test  :   1200  (75 ori × 8 strains × 2 noise)
  Total :  35200

[4/5] Simulating 35200 patterns...
    50/500 orientations  |    8.6%  |  21s elapsed
   100/500 orientations  |   19.5%  |  35s elapsed
   150/500 orientations  |   28.8%  |  46s elapsed
   200/500 orientations  |   38.7%  |  59s elapsed
   250/500 orientations  |   49.4%  |  72s elapsed
   300/500 orientations  |   59.1%  |  84s elapsed
   350/500 orientations  |   68.6%  |  96s elapsed
   400/500 orientations  |   79.5%  |  109s elapsed
   450/500 orientations  |   89.7%  |  122s elapsed
   500/500 orientations  |  100.0%  |  135s elapsed

[5/5] Writing HDF5 → /Users/jfj3094/Documents/FP-